## Carga de librerias y de la base de datos

In [ ]:
import pandas as pd #permite trabajar con excel
import numpy as np #para trabajar con matematicas
import random #generar numeros aleatorios
import copy

In [ ]:
#cargamos los datos de consumos REALES
datosConsumoReal = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/ConsumoReal(MEDICAMENTOS)2022-2024.xlsx')

datosConsumoReal.head()
datosConsumoReal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Código         390 non-null    object
 1   Desripción     390 non-null    object
 2   Concentración  390 non-null    object
 3   2022           390 non-null    int64 
 4   2023           390 non-null    int64 
 5   2024           390 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 18.4+ KB


In [ ]:
#cargamos los datos de PRECIOS de diferentes fuentes
datosPrecios = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Precios(MEDICAMENTOS)2022-2024Completo.xlsx')
datosPrecios.head()
datosPrecios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Código                    390 non-null    object 
 1   Desripción                390 non-null    object 
 2   Concentración             390 non-null    object 
 3   Costo2022 (SU)            390 non-null    int64  
 4   Costo2023 (SU)            390 non-null    float64
 5   Costo2024 (SU)            390 non-null    float64
 6   Precio LINAME Junio 2022  381 non-null    float64
 7   Precio LINAME Sep 2023    382 non-null    float64
 8   Precio LINAME Junio 2024  390 non-null    float64
 9   Precio LINAME Nov. 2024   382 non-null    float64
 10  Precio LINAME Abril 2025  382 non-null    float64
 11  Precio LINAME Junio 2025  382 non-null    float64
dtypes: float64(8), int64(1), object(3)
memory usage: 36.7+ KB


In [ ]:
#confirmamos que la lista de cantidades con la lista de precios esten ordenadas
#cada medicamento corresponde a su precio
print(datosConsumoReal.loc[335])
print(datosPrecios.loc[335])


Código                                          V-08-03
Desripción                             CONTRASTE IODADO
Concentración    Según disponibilidad (100 ml o 200 ml)
2022                                                 19
2023                                                  0
2024                                                  0
Name: 335, dtype: object
Código                                                     V-08-03
Desripción                                        CONTRASTE IODADO
Concentración               Según disponibilidad (100 ml o 200 ml)
Costo2022 (SU)                                                   0
Costo2023 (SU)                                                 0.0
Costo2024 (SU)                                                 0.0
Precio LINAME Junio 2022                                    312.26
Precio LINAME Sep 2023                                      312.26
Precio LINAME Junio 2024                                    312.26
Precio LINAME Nov. 2024             

In [ ]:
#cargo los datos del POA 2024
datosPoa = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/POA(IMEDICAMENTOS) 2024.xlsx')
datosPoa.head()
datosPoa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Codigo               390 non-null    object 
 1   Detalle              390 non-null    object 
 2   Unidad               349 non-null    object 
 3   Cant. Inicial        346 non-null    float64
 4   Cant. Entrada        346 non-null    float64
 5   Imp. Entrada         346 non-null    float64
 6   Cant. Salida         346 non-null    float64
 7   CONSUMO ANUAL 2023   346 non-null    float64
 8   Imp. Salida          346 non-null    float64
 9   SALDO  FINAL 2023    346 non-null    float64
 10  Imp/Unit Saldo       346 non-null    float64
 11  Importe Total Saldo  346 non-null    float64
 12  CPM                  346 non-null    float64
 13  CATIDAD A PEDIR      390 non-null    float64
 14  COSTO UNITARIO       390 non-null    float64
 15  TOTAL COSTO          390 non-null    flo

In [ ]:
#cargo los datos inventario
datosInventario = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Inventarios.xlsx')
datosInventario.head()
datosInventario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Código  390 non-null    object
 1   2021    390 non-null    int64 
 2   2022    390 non-null    int64 
 3   2023    390 non-null    int64 
 4   2024    390 non-null    int64 
 5   2025    390 non-null    int64 
dtypes: int64(5), object(1)
memory usage: 18.4+ KB


##PARAMETROS




1. TP		= Techo presupuestario
2. N		= Número total de medicamentos i
3. PRAi	= Precio referencial AGEMED para el medicamento i
4. PREi	= Precio referencial externo para el medicamento i
5. zi		= Cantidad comprada del producto i en el periodo anterior
6. yi		= Taza de crecimiento o decrecimiento del producto i


In [ ]:
#Para el TP sumamos todo los registros de "TOTAL COSTO" de datosPOA
TP = datosPoa['TOTAL COSTO'].sum()
print("Techo presupuestario TP = ",np.round(TP,2))


Techo presupuestario TP =  2283910.22


In [ ]:
#para N obtenemos la cantidad de variables en datosConsumoReal
N = datosConsumoReal.shape[0]
print(N)

390


In [ ]:
#para el PRA obtenemos el precio LINAME 2024 de datosPrecio
#redondeamos a 2 decimales
PRA = datosPrecios['Precio LINAME Junio 2024'] ### preguntar QUE PRECIOS corresponden ###
PRA = np.round(PRA,2)
print(PRA)
#observacion: hay medicamentos que no tienen precio en LINAME (solucion reemplace del costo del SU)
#verificacion
print(np.isnan(PRA).any())

0      12.57
1       0.57
2       0.40
3       1.82
4      14.01
       ...  
385    81.16
386    13.28
387     3.18
388     3.30
389     8.12
Name: Precio LINAME Junio 2024, Length: 390, dtype: float64
False


In [ ]:
#Para el PRE tomamos los valores PRA y le sumamos un 25% a su precio
PRE = PRA * 1.25
PRE = np.round(PRE,2)
print(PRE)
#Aqui se puede cargar la base de datos con PRE reales!
np.isnan(PRE).any()

0       15.71
1        0.71
2        0.50
3        2.28
4       17.51
        ...  
385    101.45
386     16.60
387      3.98
388      4.12
389     10.15
Name: Precio LINAME Junio 2024, Length: 390, dtype: float64


np.False_

In [ ]:
#para z sacamos el consumo REAL de las gestiones que necesitamos
z = datosConsumoReal[[2022,2023,2024]] ### PREGUNTAR SI ES CANTIDADES REALES O CANTIDADES PEDIDAS EN POA
print(z)
#para poder realizar calculos matematicos no vamos a trabajar con dataframes
#z convertido en array
z = np.array([datosConsumoReal[2022].values, datosConsumoReal[2023].values, datosConsumoReal[2024].values])
print(z)

      2022   2023   2024
0      976    980    928
1    60271  52020  52824
2     2992   1577   2246
3      524    372    267
4     2591   2807   3244
..     ...    ...    ...
385      0      0      0
386      0      0      0
387      0      0      0
388      0      0      0
389      0      0      0

[390 rows x 3 columns]
[[  976 60271  2992 ...     0     0     0]
 [  980 52020  1577 ...     0     0     0]
 [  928 52824  2246 ...     0     0     0]]


In [ ]:
#Si : Saldo total del sku i en el periodo t
S = datosInventario[[2022,2023,2024]] # Ajusta el nombre de la columna según tu Excel
print(S)
#S = np.nan_to_num(S) # Reemplaza valores vacíos (NaN) por 0 para evitar errores en el cálculo
S = np.array([datosInventario[2022].values, datosInventario[2023].values, datosInventario[2024].values])
print(S[2])

      2022  2023   2024
0      259    79    114
1    17098  6378  17574
2     1037   960    214
3      221   129    180
4      484   997    357
..     ...   ...    ...
385      2     4      9
386      9     3      7
387    130     0      0
388      0     0      0
389     11    11      0

[390 rows x 3 columns]
[  114 17574   214   180   357    47  1050   168 10756   518   475    31
   575   133    27  6702   106   900    32   407    15     0    17    75
   930    14   186   368   387  4217    56    22 13745    16     1   195
    10   875  6300   702  5132    38     0   200     0  2222     0     0
     0    20  2447  9780     3    61    11   174  1013    94  2057    13
    29   200     0  2110     0   461     0     5   124   337     0    50
     6    69    48    38     0    57    17    12   254   377     2     8
    96    54   306   296  2214     0     0   192    38     0     0    20
   460  1769   146  1683   215   284     0     0    75  3080  3654  2119
  1369  3858  2066 24292  7731 

In [ ]:
#	CPMi	: Consumo promedio mensual del sku i
# calculando sobre el último año de consumo (2024) dividido entre 12
CPM = z[2] / 12
#print(CPM)

In [ ]:
# MEDi	: Meses de existencia disponibles del SKU i
# Usamos np.divide para evitar errores de división por cero
MED = np.divide(S[2], CPM, out=np.zeros_like(S[2], dtype=float), where=CPM!=0)
print(np.shape(MED))
# LSMEDi	: Límite superior de meses de existencia disponibles del SKU i
# Este es un valor de gestión, 12 meses para esta prueba
LSMED = 12

(390,)


In [ ]:
# Definimos la variable P_i^t (Binaria)
#P es variable de desicion o es parámetro?
P = (MED < LSMED).astype(int)
print(P)

[1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0
 1 1 1 1 0 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1
 1 1 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1
 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1
 1 0 1 1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 0 1 1 0 1 1 1 0 1 0 1 1 1 1 0 1 1
 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0
 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1
 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


### Matemáticas para el cálculo de **Tasa de crecimiento o decrecimiento**
Cálculo del ajuste por uso (AU):

AU = (ConsumoReal_t - ConsumoReal_t-1) / ConsumoReal_t-1

Con AU generamos y_i

y_i=(1 + AU)

In [ ]:
#realizamos el calculo de la tasa tomando en cuenta
#cantidades gestion 2022 y gestion 2023 (usamos indices para indicar la gestion)
AU = (z[1]-z[0])/z[0]
#Existe problema con la division entre cero:
#infinito = 5/0 (division de cualquier numero entre cero)
#nan = 0/0     (division de cero entre cero)
print(AU[350:375])
print(datosConsumoReal.iloc[360])


[-1.         -1.         27.         -0.83333333  1.         -1.
  1.         -1.         -1.          2.12380952         inf         inf
         inf         inf         inf         inf         inf         inf
         inf         nan         nan         nan         nan         nan
         nan]
Código                   C-03-03
Desripción       ESPIRONOLACTONA
Concentración              25 mg
2022                           0
2023                        1128
2024                        6543
Name: 360, dtype: object


/tmp/ipython-input-1405803731.py:3: RuntimeWarning: divide by zero encountered in divide
  AU = (z[1]-z[0])/z[0]
/tmp/ipython-input-1405803731.py:3: RuntimeWarning: invalid value encountered in divide
  AU = (z[1]-z[0])/z[0]


In [ ]:

#para el caso de valores nan (Not a number) asumimos que no existe crecimiento ni decrecimiento (0.0)
### PREGUNTAR ESTE CASO EN PARTICULAR ###
AU = np.where(np.isnan(AU), 0, AU)
#para el caso especial INFINITO asumimos crecimiento del 100%
#limpiamos valores inf (infinitos) y remplazamos por 1.0
AU = np.where(~np.isfinite(AU), 0, AU)
#verificamos
print(AU[350:375])


[-1.         -1.         27.         -0.83333333  1.         -1.
  1.         -1.         -1.          2.12380952  0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.        ]


In [ ]:
#finalmente obtenemos y_i
y = (1+AU)
print(y[350:375])

[ 0.          0.         28.          0.16666667  2.          0.
  2.          0.          0.          3.12380952  1.          1.
  1.          1.          1.          1.          1.          1.
  1.          1.          1.          1.          1.          1.
  1.        ]


## Cálculo cantidad proyectada:
CantidadProyectada = zi × yi

Representa la cantidad justificable para ser adquirida en el nuevo año, y se utilizará como punto de referencia para que se muevan wi y xi en la metaheuristica.

In [ ]:
CantidadProyectada = np.ceil(z[1]*y).astype(int) #redondeado al entero superior
CantidadProyectada = CantidadProyectada*P #CANTIDAD PROYECTADA AJUSTADA
print("consumom2023 vs proyectado 2024 vs Inventario 2023")
for i, j, k in zip(z[1], CantidadProyectada, np.array(datosInventario[2023])):
  print(i,"\t \t", j, "\t \t", k)

consumom2023 vs proyectado 2024 vs Inventario 2023
980 	 	 985 	 	 79
52020 	 	 44899 	 	 6378
1577 	 	 832 	 	 960
372 	 	 265 	 	 129
2807 	 	 3042 	 	 997
49 	 	 49 	 	 16
584 	 	 0 	 	 433
478 	 	 572 	 	 194
25638 	 	 22719 	 	 5446
1297 	 	 998 	 	 1093
1670 	 	 1987 	 	 243
62 	 	 0 	 	 55
1700 	 	 1476 	 	 453
476 	 	 463 	 	 98
136 	 	 163 	 	 31
19552 	 	 13046 	 	 11726
571 	 	 810 	 	 47
3185 	 	 5847 	 	 610
162 	 	 144 	 	 114
276 	 	 492 	 	 242
5913 	 	 5261 	 	 1193
55 	 	 37 	 	 82
9 	 	 0 	 	 22
375 	 	 287 	 	 37
495 	 	 0 	 	 0
77 	 	 92 	 	 32
826 	 	 920 	 	 680
2259 	 	 2341 	 	 191
7654 	 	 8391 	 	 1600
15931 	 	 15232 	 	 3005
295 	 	 330 	 	 62
69 	 	 55 	 	 28
83532 	 	 83124 	 	 18420
145 	 	 270 	 	 12
2710 	 	 7434 	 	 0
1514 	 	 1582 	 	 90
22 	 	 0 	 	 8
1132 	 	 816 	 	 550
14399 	 	 15485 	 	 2512
2591 	 	 2194 	 	 928
14046 	 	 14240 	 	 3197
23 	 	 0 	 	 48
0 	 	 0 	 	 0
317 	 	 0 	 	 12
589 	 	 516 	 	 0
19937 	 	 23976 	 	 3700
3421 	 	 3040 	 	 

In [ ]:
#vamos a multiplicar la cantidad proyectada por los PRA 2024
#observamos cuanto costaria pedir la cantidad proyectada para 2024
pruebaCosto = CantidadProyectada*PRA
print("Proyectado2024: ", np.sum(pruebaCosto))
print("TP2024:         ", TP)
cc = z[2]*PRA
print("Real2024:       ", np.sum(cc))

Proyectado2024:  5732658.36
TP2024:          2283910.2177857975
Real2024:        3892971.9699999997


## Funciones
ya tenemos la capacidad de referencia ahora pasamo a generar la población inicial utilizando las funciones ya definidas

In [ ]:
def calcular_fitness(wi, xi, TP, PRA, PRE, S, P):
    # 1. Calculamos el costo de las adquisiciones
    costo_planificado = np.sum(wi * PRA)
    costo_extra = np.sum(xi * PRE)
    costo_total_compra = costo_planificado + costo_extra

    # 2. Calculamos la valoración del inventario condicionado
    valor_inventario_ajustado = np.sum(S * PRA * P)

    # 3. Aplicamos la nueva función objetivo (CT)
    CT = costo_total_compra - TP - valor_inventario_ajustado

    return CT, costo_planificado, costo_extra, valor_inventario_ajustado

def verificadorCP(costo_planificado, TP, S, PRA):
    # 1. Calculamos el valor monetario del inventario actual
    valor_inventario_actual = np.sum(S * PRA)

    # 2. Calculamos el presupuesto disponible para nuevas compras
    presupuesto_limite = TP - valor_inventario_actual

    # 3. Verificación lógica
    if costo_planificado > presupuesto_limite:
        return False
    else:
        return True



def generar_poblacion_inicial(N_pop, CantidadProyectada, N, TP, PRA, PRE):

    poblacion = []
    rng = np.random.default_rng()

    while len(poblacion) < N_pop:

      proporcion = rng.uniform(0.6, 0.95, size=N) #
      #shares = rng.uniform(0.0, 1.0, size=N)
      w = np.floor(proporcion * CantidadProyectada).astype(int)
      x = np.array(CantidadProyectada - w).astype(int)

      #calculo fitness
      fitness, CP, CE = calcular_fitness(w, x, TP, PRA, PRE)
      if verificadorCP(CP, TP) == True:
        poblacion.append([w, x])

    return poblacion





In [ ]:
poblacion = generar_poblacion_inicial(3, CantidadProyectada, N, TP, PRA, PRE)
print(np.shape(poblacion))
print(np.shape(poblacion[1]))

(3, 2, 390)
(2, 390)


In [ ]:
w = poblacion[0][0]
x = poblacion[0][1]
fitness, Costo_plan, Costo_extra = calcular_fitness(w, x, TP, PRA, PRE)
print(fitness)

977917.4544284055


In [ ]:
#FUNCIONES BWO

#-----------PROCREACION
def procreacion(parent1, parent2, CantidadProyectada, N, TP, PRA, PRE, CR, P):

    # Extraer w y x de cada parent
    w1, x1 = np.array(parent1[0]), np.array(parent1[1])
    w2, x2 = np.array(parent2[0]), np.array(parent2[1])

    CantidadProyectada_A= CantidadProyectada*P #cantidad proyectada ajustada con P


    # Verificar si N es par
    Nvar = N if N % 2 == 0 else N - 1

    children = []

    # Realizar la procreación (cruce)
    while len(children) < Nvar:
        alfa = np.random.uniform(0.0, 1.0)

        # Cruce para w (cantidades anticipadas)
        w_child1 = np.round(alfa * w1 + (1 - alfa) * w2).astype(int)
        w_child2 = np.round(alfa * w2 + (1 - alfa) * w1).astype(int)

        # Cruce para x (cantidades extras)
        x_child1 = np.round(alfa * x1 + (1 - alfa) * x2).astype(int)
        x_child2 = np.round(alfa * x2 + (1 - alfa) * x1).astype(int)

        # Ajustar para que w + x = CantidadProyectada*P
        # Priorizar mantener w y ajustar x
        for i in range(N):
            total = w_child1[i] + x_child1[i]
            if total != CantidadProyectada_A[i]:
                x_child1[i] = max(0, CantidadProyectada_A[i] - w_child1[i])
                w_child1[i] = CantidadProyectada_A[i] - x_child1[i]

            total = w_child2[i] + x_child2[i]
            if total != CantidadProyectada_A[i]:
                x_child2[i] = max(0, CantidadProyectada_A[i] - w_child2[i])
                w_child2[i] = CantidadProyectada[i] - x_child2[i]


        #verificamos que se cumpla restriccion CP<=TP
        fitness_child1, CPchild1, CEchild1 = calcular_fitness(w_child1, x_child1, TP, PRA, PRE)
        if verificadorCP(CPchild1) == True:
          # Agregar hijos a la lista
          children.append([w_child1, x_child1])

        fitness_child2, CPchild2, CEchild2 = calcular_fitness(w_child2, x_child2, TP, PRA, PRE)
        if verificadorCP(CPchild2) == True:
          # Agregar hijos a la lista
          children.append([w_child2, x_child2])

    # Calcular fitness de los hijos
    fit_children = []
    for child in children:
        fitness, CP, CE = calcular_fitness(child[0], child[1], TP, PRA, PRE)
        fit_children.append(fitness)

    # Convertir a arrays numpy para ordenar
    fit_children = np.array(fit_children)
    children = np.array(children, dtype=object)

    # Ordenar por fitness (ascendente, asumiendo que menor fitness es mejor)
    sorted_indices = np.argsort(fit_children)
    fit_children = fit_children[sorted_indices]
    children = children[sorted_indices]

    # Canibalismo de los hijos
    ns = int(len(children) * CR)  # Número de sobrevivientes
    ns = max(1, ns)  # Asegurar al menos 1 sobreviviente

    pop2 = children[:ns].tolist()
    fit_children = fit_children[:ns]

    return pop2, fit_children


#--------------MUTACION-------
def mutacion_gaussiana(pop1, PM, CantidadProyectada, N, TP, PRA, PRE, sigma=7):

    nm = int(len(pop1) * PM)  # Número de mutaciones
    nm = max(1, nm)  # Asegurar al menos 1 mutación

    pop3 = []
    fit_mutation = []

    while len(pop3) < nm:
        # Seleccionar un individuo aleatorio para mutar
        individuo = random.choice(pop1)
        w_mut = individuo[0].copy()
        x_mut = individuo[1].copy()

        # Mutación gaussiana en w (cantidades anticipadas)
        for i in range(N):
            delta = int(round(np.random.normal(0, sigma)))

            # Aplicar mutación a w
            w_nuevo = w_mut[i] + delta

            # Asegurar que w no sea negativo and no exceda la capacidad proyectada
            w_nuevo = max(0, min(w_nuevo, CantidadProyectada[i]))

            # Ajustar w and x para mantener la restricción w + x = CantidadProyectada
            w_mut[i] = w_nuevo
            x_mut[i] = CantidadProyectada[i] - w_mut[i]

        #verificamos CP<=TP antes de guardar
        fitness, CP, CE = calcular_fitness(w_mut, x_mut, TP, PRA, PRE)
        if verificadorCP(CP) == True:
          pop3.append([w_mut, x_mut])

    # Calcular el fitness de los individuos mutados
    for individuo in pop3:
        fitness, CP,CE = calcular_fitness(individuo[0], individuo[1], TP, PRA, PRE)
        fit_mutation.append(fitness)

    fit_mutation = np.array(fit_mutation)

    return pop3, fit_mutation

## MAIN

In [ ]:
#PARAMETROS BWO
PR = 0.8  # procreation rate
CR = 0.4  # tasa canibalismo
PM = 0.8  # tasa de mutacion
N_Pop = 50  # numero de widows iniciales
N_Iter = 100  # numero de iteraciones

#para poder trabajar necesitamos transformar todos los dataframes a arrays de numpy
PRA = np.array(PRA)
PRE = np.array(PRE)
CantidadProyectada = np.array(CantidadProyectada)

# Generar población inicial
Pop = generar_poblacion_inicial(N_Pop, CantidadProyectada, N, TP, PRA, PRE)

# Calcular fitness inicial
Fit_pop = []
for individuo in Pop:
    fitness, plan, extra = calcular_fitness(individuo[0], individuo[1], TP, PRA, PRE)
    Fit_pop.append(fitness)
Fit_pop = np.array(Fit_pop)

# Ordenar población inicial por fitness (ascendente: menor fitness es mejor)
sorted_indices = np.argsort(Fit_pop)
Fit_pop = Fit_pop[sorted_indices]
Pop = [Pop[i] for i in sorted_indices]

# Almacenar historial
fits = []
wis = []

# Algoritmo BWO
for iter in range(N_Iter):
    print(f"--------Iteración: {iter+1}/{N_Iter} - Mejor Fitness: {Fit_pop[0]:.2f}")

    # Generación de población de PADRES (Pop1)
    nr = int(N_Pop * PR)  # número de reproducciones
    nr = max(2, nr)  # Asegurar al menos 2 padres para reproducción
    Pop1 = copy.deepcopy(Pop[:nr])

    # Seleccionar 2 padres aleatorios de Pop1
    if len(Pop1) >= 2:
        indices_padres = random.sample(range(len(Pop1)), k=2)
        Parent1 = Pop1[indices_padres[0]]
        Parent2 = Pop1[indices_padres[1]]

        # Calcular fitness de los padres para canibalismo
        fit_parent1, planp1, extp1 = calcular_fitness(Parent1[0], Parent1[1], TP, PRA, PRE)
        fit_parent2, planp2, extp2 = calcular_fitness(Parent2[0], Parent2[1], TP, PRA, PRE)

        # Canibalismo de los PADRES (eliminar el padre más débil)
        # Encontrar el índice del padre más débil en Pop
        idx_debil = None
        for idx, individuo in enumerate(Pop):
            if (np.array_equal(individuo[0], Parent1[0]) and
                np.array_equal(individuo[1], Parent1[1]) and
                fit_parent1 < fit_parent2):
                idx_debil = idx
                break
            elif (np.array_equal(individuo[0], Parent2[0]) and
                  np.array_equal(individuo[1], Parent2[1]) and
                  fit_parent2 < fit_parent1):
                idx_debil = idx
                break

        if idx_debil is not None:
            Pop.pop(idx_debil)
            Fit_pop = np.delete(Fit_pop, idx_debil)
    else:
        # Si no hay suficientes padres, seleccionar los dos primeros
        Parent1 = Pop[0]
        Parent2 = Pop[1] if len(Pop) > 1 else Pop[0]

    #-------------Procreación------------------------------
    Pop2, Fit_children = procreacion(Parent1, Parent2, CantidadProyectada,
                                     N, TP, PRA, PRE, CR)

    #--------------Mutación-------------------------------
    sigma = 10 * (1.1 - iter / N_Iter)
    Pop3, Fit_mutation = mutacion_gaussiana(Pop1, PM, CantidadProyectada,
                                           N, TP, PRA, PRE, sigma)

    #--------------Actualización de la población------------------------
    # Combinar todas las poblaciones
    Pop = Pop + Pop2 + Pop3
    Fit_pop = np.concatenate((Fit_pop, Fit_children, Fit_mutation))

    # Ordenar por fitness (ascendente: menor fitness es mejor)
    sorted_indices = np.argsort(Fit_pop)
    Fit_pop = Fit_pop[sorted_indices]
    Pop = [Pop[i] for i in sorted_indices]

    # Recortar la población para mantener N_Pop individuos
    Pop = copy.deepcopy(Pop[:N_Pop])
    Fit_pop = copy.deepcopy(Fit_pop[:N_Pop])

    # Guardar fitness y widow de la mejor solución
    fits.append(Fit_pop[0])
    wis.append(copy.deepcopy(Pop[0]))

print("__________________________________________________________")
print(f"MEJOR COSTO ENCONTRADO: {fits[-1]:.2f} $")
print("MEJOR WIDOW:")
print(f"  w (planificacidas): {wis[-1][0]}")
print(f"  x (extras): {wis[-1][1]}")
print(f"  cantidad proyectada: {CantidadProyectada.tolist()}")


# Calcular costos de la mejor solución
w_mejor = wis[-1][0]
x_mejor = wis[-1][1]
costo_anticipado = sum(w_mejor[i] * PRA[i] for i in range(N))
costo_extra = sum(x_mejor[i] * PRE[i] for i in range(N))
costo_total = costo_anticipado + costo_extra

print(f"\nDetalle de costos:")
print(f"  Costo planificado: {costo_anticipado:.2f} Bs")
print(f"  Costo extra: {costo_extra:.2f} Bs")
#print(f"  Costo total: {costo_total:.2f} $")
print(f"  Presupuesto (TP): {TP:.2f} Bs")
#print(f"  Ahorro: {TP - costo_total:.2f} $")
print("__________________________________________________________")
print(f"Diferencia TP - Costo Planificado : {round((TP-costo_anticipado),2)}")

--------Iteración: 1/100 - Mejor Fitness: 1700347.83
--------Iteración: 2/100 - Mejor Fitness: 1700347.83
--------Iteración: 3/100 - Mejor Fitness: 1700347.83
--------Iteración: 4/100 - Mejor Fitness: 1700347.83
--------Iteración: 5/100 - Mejor Fitness: 1700347.83
--------Iteración: 6/100 - Mejor Fitness: 1700347.83
--------Iteración: 7/100 - Mejor Fitness: 1700308.15
--------Iteración: 8/100 - Mejor Fitness: 1700308.15
--------Iteración: 9/100 - Mejor Fitness: 1700308.15
--------Iteración: 10/100 - Mejor Fitness: 1700308.15
--------Iteración: 11/100 - Mejor Fitness: 1700109.30
--------Iteración: 12/100 - Mejor Fitness: 1699999.55
--------Iteración: 13/100 - Mejor Fitness: 1699999.55
--------Iteración: 14/100 - Mejor Fitness: 1699999.55
--------Iteración: 15/100 - Mejor Fitness: 1699999.55
--------Iteración: 16/100 - Mejor Fitness: 1699999.55
--------Iteración: 17/100 - Mejor Fitness: 1699999.55
--------Iteración: 18/100 - Mejor Fitness: 1699999.55
--------Iteración: 19/100 - Mejor Fit

In [ ]:
print("__________________________________________________________")
print(f"MEJOR COSTO ENCONTRADO HASTA AHORA: {fits[-1]:.2f} $")
print("MEJOR WIDOW:")
print(f"  w (planificacidas): {wis[-1][0]}")
print(f"  x (extras): {wis[-1][1]}")
print(f"  cantidad proyectada: {CantidadProyectada.tolist()}")


# Calcular costos de la mejor solución
w_best = wis[-1][0]
x_best = wis[-1][1]
costo_anticipado = sum(w_mejor[i] * PRA[i] for i in range(N))
costo_extra = sum(x_mejor[i] * PRE[i] for i in range(N))
costo_total = costo_anticipado + costo_extra

print(f"\nEl mejor hasta ahora:")
print(f"  Costo planificado: {costo_anticipado:.2f} Bs")
print(f"  Costo extra: {costo_extra:.2f} Bs")
#print(f"  Costo total: {costo_total:.2f} $")
print(f"  Presupuesto (TP): {TP:.2f} Bs")
#print(f"  Ahorro: {TP - costo_total:.2f} $")
print("__________________________________________________________")
print(f"Diferencia TP - Costo Planificado : {round((TP-costo_anticipado),2)}")

__________________________________________________________
MEJOR COSTO ENCONTRADO HASTA AHORA: 1129130.77 $
MEJOR WIDOW:
  w (planificacidas): [872, 33882, 758, 175, 1941, 30, 291, 474, 16106, 805, 1230, 77, 1377, 266, 120, 11376, 694, 4885, 97, 299, 4720, 31, 0, 209, 268, 65, 698, 2190, 7682, 11277, 267, np.float64(54.0), 76635, 260, 6697, 1045, 5, 568, 10886, 1684, 10952, 4, 0, 186, 422, 16726, 2148, 32357, 0, 41, 18266, 24106, 437, 26, 10, 976, 8485, 63, 5261, 153, 157, 997, 0, 3377, 25, 2788, 185, np.float64(53.0), 269, 493, 45, 93, 41, 315, 129, 8, 5, 83, 0, 14, 528, 1060, 0, 61, 311, 125, 1299, 385, 4442, 27, 0, 378, 136, 244, 2, 183, 1506, 2228, 210, 5050, 170, 327, 93, 819, 848, 19525, 18811, 9811, 7138, 19308, 8745, 124727, 13430, 6232, 6562, 45313, 30, 206, 0, 39, 11, 83, 0, 1, 67, 94, 85, 37, 79, 45, 0, 0, 396, 66, 185, 0, 84, 0, 253, 82, 1625, 4, 2, 183, 87, 123, 170, 17, 0, 9, 4942, 44, 0, 4651, 389, np.float64(65.0), 33, 6583, 1938, 38154, 0, 0, 723, 1095, 145, 459, 33, 8

#prueba

In [ ]:

import numpy as np
sorteo = np.random.choice(np.arange(0, 9), size=(3, 3), replace=False)
print(sorteo)

[[6 0 2]
 [3 1 4]
 [5 8 7]]
